# Alerts and Action Groups for Microsoft Foundry

Monitoring data is only useful if it triggers action. This notebook creates alert rules that fire when your Foundry agents cross critical thresholds — and action groups that determine who gets notified.

**What you will learn:**
- Create an Action Group with email notification
- Create a log search alert that fires on agent errors
- Create a metric alert that fires on high API latency
- Verify and test your alert configuration

**Prerequisites:**
- Azure CLI installed and signed in (`az login`)
- Application Insights configured and receiving Foundry telemetry
- An Azure AI Services resource with active model deployments
- **Contributor** role on the resource group

In [ ]:
%pip install python-dotenv -q

## Configuration

Copy `.env.example` to `.env` and fill in your values:

| Variable | Where to find it |
|---|---|
| `RESOURCE_GROUP` | Azure Portal → Resource groups |
| `AI_SERVICES_RESOURCE_NAME` | Your Azure AI Services resource name |
| `LOG_ANALYTICS_WORKSPACE_NAME` | The Log Analytics workspace receiving your telemetry |
| `ALERT_EMAIL` | Email address for alert notifications |

In [ ]:
import os
import subprocess
import json
from dotenv import load_dotenv

load_dotenv(override=True)

RESOURCE_GROUP = os.getenv("RESOURCE_GROUP")
AI_SERVICES_NAME = os.getenv("AI_SERVICES_RESOURCE_NAME")
WORKSPACE_NAME = os.getenv("LOG_ANALYTICS_WORKSPACE_NAME")
ALERT_EMAIL = os.getenv("ALERT_EMAIL")

# Verify Azure CLI login
result = subprocess.run(["az", "account", "show", "-o", "json"],
                        capture_output=True, text=True)
if result.returncode == 0:
    acct = json.loads(result.stdout)
    print(f"\u2705 Signed in as {acct['user']['name']}")
else:
    print("\u274c Not signed in. Run 'az login' in your terminal first.")

print(f"\nResource Group:  {RESOURCE_GROUP}")
print(f"AI Services:     {AI_SERVICES_NAME}")
print(f"Workspace:       {WORKSPACE_NAME}")
print(f"Alert Email:     {ALERT_EMAIL}")

In [ ]:
# Resolve resource IDs needed for alert configuration

# Log Analytics workspace ID
result = subprocess.run(
    ["az", "monitor", "log-analytics", "workspace", "show",
     "--resource-group", RESOURCE_GROUP,
     "--workspace-name", WORKSPACE_NAME,
     "-o", "json"],
    capture_output=True, text=True
)
if result.returncode != 0:
    raise RuntimeError(f"Failed to resolve Log Analytics workspace:\n{result.stderr}")
workspace = json.loads(result.stdout)
WORKSPACE_ID = workspace["id"]

# AI Services resource ID
result = subprocess.run(
    ["az", "cognitiveservices", "account", "show",
     "--resource-group", RESOURCE_GROUP,
     "--name", AI_SERVICES_NAME,
     "-o", "json"],
    capture_output=True, text=True
)
if result.returncode != 0:
    raise RuntimeError(f"Failed to resolve AI Services resource:\n{result.stderr}")
ai_resource = json.loads(result.stdout)
AI_SERVICES_ID = ai_resource["id"]

print(f"\u2705 Workspace ID resolved")
print(f"\u2705 AI Services ID resolved")

## Step 1: Create an Action Group

An Action Group defines **who gets notified and how** when an alert fires. You can combine multiple notification channels — email, SMS, webhook, Azure Function, Logic App — in a single group.

For this demo, we create an Action Group with email notification. In production, you would also add a webhook to your incident management tool (PagerDuty, ServiceNow, Slack).

In [ ]:
ACTION_GROUP_NAME = "foundry-alerts"
ACTION_GROUP_SHORT = "fndryalert"

result = subprocess.run(
    ["az", "monitor", "action-group", "create",
     "--resource-group", RESOURCE_GROUP,
     "--name", ACTION_GROUP_NAME,
     "--short-name", ACTION_GROUP_SHORT,
     "--action", "email", "admin-email", ALERT_EMAIL,
     "-o", "json"],
    capture_output=True, text=True
)

if result.returncode == 0:
    action_group = json.loads(result.stdout)
    ACTION_GROUP_ID = action_group["id"]
    print(f"\u2705 Action Group created: {ACTION_GROUP_NAME}")
    print(f"   Short name:  {ACTION_GROUP_SHORT}")
    print(f"   Email:       {ALERT_EMAIL}")
    print(f"\n   Resource ID: {ACTION_GROUP_ID}")
else:
    print(f"\u274c Failed to create Action Group:\n{result.stderr}")

## Step 2: Log Search Alert — Agent Error Rate

This alert fires when **more than 5 failed AI model calls** occur within a 1-hour window. It queries the `AppDependencies` table in your Log Analytics workspace — the same table the KQL demos use.

**How it works:**
- The KQL query counts failed model calls (`Success == false`) in each evaluation window
- Azure Monitor runs this query every 15 minutes against the last hour of data
- If the count exceeds 5, the alert fires and notifies the Action Group

**Severity 2 (Warning):** High enough to warrant investigation, not yet critical.

In [ ]:
ERROR_ALERT_NAME = "foundry-agent-error-rate"

# The KQL query for the alert condition.
# Scheduled-query alerts use a named placeholder: --condition references the
# placeholder and applies the aggregation (count of returned rows); --condition-query
# binds Placeholder_1 to the KQL that returns the matching (failed) rows.
error_query = (
    "AppDependencies "
    "| where Properties has 'gen_ai.system' "
    "| where Success == false"
)

result = subprocess.run(
    ["az", "monitor", "scheduled-query", "create",
     "--name", ERROR_ALERT_NAME,
     "--resource-group", RESOURCE_GROUP,
     "--scopes", WORKSPACE_ID,
     "--condition-query", f"Placeholder_1={error_query}",
     "--condition", "count 'Placeholder_1' > 5",
     "--window-size", "1h",
     "--evaluation-frequency", "15m",
     "--severity", "2",
     "--action-groups", ACTION_GROUP_ID,
     "--description", "Fires when more than 5 AI model calls fail within 1 hour",
     "-o", "json"],
    capture_output=True, text=True
)

if result.returncode == 0:
    alert = json.loads(result.stdout)
    print(f"✅ Log search alert created: {ERROR_ALERT_NAME}")
    print(f"   Query:      Failed model calls (Success == false)")
    print(f"   Threshold:  FailedCalls > 5 per hour")
    print(f"   Frequency:  Evaluated every 15 minutes")
    print(f"   Severity:   2 (Warning)")
else:
    print(f"❌ Failed to create alert:\n{result.stderr}")
    print("\nNote: If the command format differs in your CLI version,")
    print("check 'az monitor scheduled-query create --help' for the exact syntax.")

## Step 3: Metric Alert — High API Latency

This alert monitors a **platform metric** directly on the AI Services resource — no KQL query needed. It fires when average API latency exceeds 10 seconds over a 5-minute window.

**Why both alert types matter:**
- **Log search alerts** (Step 2) are flexible — you write any KQL query. Best for application-level signals (error patterns, token spikes, specific agent behaviors).
- **Metric alerts** (this step) are faster — they evaluate against pre-aggregated metrics. Best for infrastructure signals (latency, throughput, throttling).

In [ ]:
LATENCY_ALERT_NAME = "foundry-high-latency"

result = subprocess.run(
    ["az", "monitor", "metrics", "alert", "create",
     "--name", LATENCY_ALERT_NAME,
     "--resource-group", RESOURCE_GROUP,
     "--scopes", AI_SERVICES_ID,
     "--condition", "avg Latency > 10000",
     "--window-size", "5m",
     "--evaluation-frequency", "1m",
     "--severity", "2",
     "--action", ACTION_GROUP_ID,
     "--description", "Fires when average API latency exceeds 10 seconds",
     "-o", "json"],
    capture_output=True, text=True
)

if result.returncode == 0:
    alert = json.loads(result.stdout)
    print(f"\u2705 Metric alert created: {LATENCY_ALERT_NAME}")
    print(f"   Metric:     Latency (average)")
    print(f"   Threshold:  > 10,000 ms (10 seconds)")
    print(f"   Frequency:  Evaluated every 1 minute")
    print(f"   Severity:   2 (Warning)")
else:
    print(f"\u274c Failed to create alert:\n{result.stderr}")
    print("\nTo see available metrics for your AI Services resource:")
    print(f"  az monitor metrics list-definitions --resource {AI_SERVICES_ID}")

## Step 4: Verify Alert Configuration

List all alerts in the resource group to confirm both rules are active.

In [ ]:
print("=" * 60)
print("LOG SEARCH ALERTS (scheduled query rules)")
print("=" * 60)

result = subprocess.run(
    ["az", "monitor", "scheduled-query", "list",
     "--resource-group", RESOURCE_GROUP,
     "-o", "json"],
    capture_output=True, text=True
)

if result.returncode == 0:
    alerts = json.loads(result.stdout)
    for a in alerts:
        enabled = a.get("enabled", True)
        status = "\u2705 Enabled" if enabled else "\u274c Disabled"
        print(f"\n  {a['name']}")
        print(f"    Status:    {status}")
        print(f"    Severity:  {a.get('severity', 'N/A')}")
        print(f"    Window:    {a.get('windowSize', 'N/A')}")
        print(f"    Frequency: {a.get('evaluationFrequency', 'N/A')}")
    if not alerts:
        print("  No log search alerts found.")
else:
    print(f"  Could not list log search alerts: {result.stderr[:200]}")

print(f"\n{'=' * 60}")
print("METRIC ALERTS")
print("=" * 60)

result = subprocess.run(
    ["az", "monitor", "metrics", "alert", "list",
     "--resource-group", RESOURCE_GROUP,
     "-o", "json"],
    capture_output=True, text=True
)

if result.returncode == 0:
    alerts = json.loads(result.stdout)
    for a in alerts:
        enabled = a.get("enabled", True)
        status = "\u2705 Enabled" if enabled else "\u274c Disabled"
        print(f"\n  {a['name']}")
        print(f"    Status:    {status}")
        print(f"    Severity:  {a.get('severity', 'N/A')}")
        print(f"    Window:    {a.get('windowSize', 'N/A')}")
        print(f"    Frequency: {a.get('evaluationFrequency', 'N/A')}")
    if not alerts:
        print("  No metric alerts found.")
else:
    print(f"  Could not list metric alerts: {result.stderr[:200]}")

## Step 5: View Available Platform Metrics

The metric alert in Step 3 used `Latency`. Your AI Services resource exposes many more metrics you can alert on. Run this cell to see the full list — useful for building additional alerts tailored to your workload.

In [ ]:
result = subprocess.run(
    ["az", "monitor", "metrics", "list-definitions",
     "--resource", AI_SERVICES_ID,
     "-o", "json"],
    capture_output=True, text=True
)

if result.returncode == 0:
    metrics = json.loads(result.stdout)
    print(f"Available metrics for {AI_SERVICES_NAME}:\n")
    for m in metrics.get("value", metrics) if isinstance(metrics, dict) else metrics:
        name = m.get("name", {}).get("value", "unknown")
        display = m.get("name", {}).get("localizedValue", name)
        unit = m.get("unit", "")
        aggs = ", ".join(m.get("supportedAggregationTypes", []))
        print(f"  {display}")
        print(f"    Metric name:   {name}")
        print(f"    Unit:          {unit}")
        print(f"    Aggregations:  {aggs}")
        print()
else:
    print(f"Could not list metrics:\n{result.stderr}")

## Summary

You created two types of alerts that cover the monitoring stack:

| Alert | Type | Signal | Threshold |
|---|---|---|---|
| Agent Error Rate | Log search | Failed model calls in `AppDependencies` | > 5 per hour |
| High API Latency | Metric | `Latency` on AI Services resource | > 10 seconds avg |

**Alert types in practice:**
- **Log search alerts** give you full KQL flexibility. Use them for agent-level signals: error patterns, unexpected tool failures, token budget violations, specific user impact.
- **Metric alerts** evaluate faster (near real-time). Use them for infrastructure signals: latency spikes, throttling, request volume drops.

**Common production alerts for Foundry agents:**
- Error rate exceeding baseline (log search)
- Token consumption exceeding daily budget (log search)
- P95 latency above SLA threshold (metric)
- Throttled requests above zero (metric — the `RateLimitedCalls` metric)
- No telemetry received in X minutes (log search — "dead agent" detection)

**Cleanup:** If these were demo alerts, you can remove them with:
```bash
az monitor scheduled-query delete --name foundry-agent-error-rate --resource-group <rg> --yes
az monitor metrics alert delete --name foundry-high-latency --resource-group <rg>
az monitor action-group delete --name foundry-alerts --resource-group <rg>
```